In [1]:
from nichenetpy.utils import (
    read_csv_cols,
)
from nichenetpy.parameter_optimization import construct_and_evaluate

from itertools import chain, repeat

import os
import requests
import pandas as pd
import session_info
import json
import numpy as np

In [2]:
network_path = os.path.normpath("./tutorial_files/model_construction/human")
if not os.path.exists(network_path):
    os.makedirs(network_path)
for filename in (
    "gr_human.csv",
    "lr_network_human.csv",
    "lr_sig_human.csv",
    "optimized_source_weights.csv",
    "annotation_data_sources.csv"
):
    file_path = os.path.join(network_path, filename)
    if not os.path.exists(file_path):
        res = requests.get(f"https://zenodo.org/records/14929618/files/{filename}")
        with open(file_path, "wb") as file:
            file.write(res.content)

In [3]:
gr_network = pd.DataFrame(read_csv_cols(os.path.join(network_path, "gr_human.csv")))
lr_network = pd.DataFrame(read_csv_cols(os.path.join(network_path, "lr_network_human.csv")))
sig_network = pd.DataFrame(read_csv_cols(os.path.join(network_path, "lr_sig_human.csv")))

In [4]:
len(set(lr_network["source"]).union(gr_network["source"], sig_network["source"]))

57

In [5]:
train_path = "D:/Data/nichenetpy/model_optimization"
with open(os.path.join(train_path, "settings_training_f1234.json"), "rb") as file:
    settings_CV = json.loads(file.read())
settings = settings_CV["settings"]

In [6]:
gr_network = gr_network[
    ((gr_network["database"] == "NicheNet_LT") & np.array([fr not in settings_CV["forbidden_ligands_nichenet"] for fr in gr_network["from"]]))
    |
    ((gr_network["database"] == "CytoSig") & np.array([fr not in settings_CV["forbidden_ligands_cytosig"] for fr in gr_network["from"]]))
]

In [7]:
eval = construct_and_evaluate(
    dict(zip(set(chain(gr_network["source"], lr_network["source"], sig_network["source"])), repeat(1))),
    lr_network,
    gr_network,
    sig_network,
    settings
)

c:\Users\victorm\Documents\nichenetpy\.hatch\Lib\site-packages\sklearn\metrics\_ranking.py:1030: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
c:\Users\victorm\Documents\nichenetpy\.hatch\Lib\site-packages\sklearn\metrics\_ranking.py:1183: UndefinedMetricWarning: No positive samples in y_true, true positive value should be meaningless
  warnings.warn(
C:\Users\victorm\Documents\nichenetpy\src\nichenetpy\metrics.py:161: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  pcc = pearsonr(response, prediction).statistic
c:\Users\victorm\Documents\nichenetpy\.hatch\Lib\site-packages\sklearn\metrics\_ranking.py:1030: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
c:\Users\victorm\Documents\nichenetpy\.hatch\Lib\site-packages\sklearn\metrics\_ranking.py:1183: UndefinedMetricWarning: No positive samples in y_true, true positive value

In [8]:
eval["performances_target_prediction"]

,setting,ligand,auroc,pearson,aupr,aupr_corrected
0,ifna_ifng_timeseries,IFNG,0.530994,0.021983,0.012009,0.004159
1,spectrum_Ifnb1,IFNB1,0.513481,0.020767,0.041388,0.009011
2,GSE6085_IL2_timeseries,IL2,0.496339,-0.002005,0.026228,0.000002
3,il20family_Il1b,IL1B,0.514967,0.030652,0.051478,0.026124
4,rat_Ifng,IFNG,0.526954,0.038939,0.044894,0.014359
...,...,...,...,...,...,...
126,dataset_WNT3A,WNT3A,0.491958,-0.004158,0.022423,-0.000473
127,dataset_INHBA,INHBA,0.521982,0.012476,0.029157,0.001465
128,dataset_IL36A,IL36A,0.505315,0.003046,0.026011,-0.000133
129,dataset_IL12A,IL12A,0.627616,0.147390,0.072237,0.046865


In [9]:
from nichenetpy.utils import extract_ligands_from_settings
ligands_evaluation = extract_ligands_from_settings(settings, combination=True)

In [23]:
from nichenetpy.parameter_optimization import _average_performances
performances_target_prediction_averaged = [
    _average_performances(ligand, eval["performances_target_prediction"])
    for ligand in ligands_evaluation
]

In [18]:
eval["performances_ligand_prediction"]

,metric,group,ligand,aupr,aupr_corrected,auroc,pearson
0,aupr,spectrum_Ifng_Tnf,"[TNF, IFNG]",0.500000,0.500000,NaN,NaN
1,aupr_corrected,spectrum_Ifng_Tnf,"[TNF, IFNG]",0.500000,0.500000,NaN,NaN
2,auroc,spectrum_Ifng_Tnf,"[TNF, IFNG]",0.500000,0.500000,NaN,NaN
3,pearson,spectrum_Ifng_Tnf,"[TNF, IFNG]",0.500000,0.500000,NaN,NaN
0,aupr,dataset_IL1A,IL1A,0.008929,-0.006944,0.112903,-0.053005
...,...,...,...,...,...,...,...
3,pearson,rat_Il1b,IL1B,0.250000,0.234127,0.983871,0.438010
0,aupr,dataset_IL10,IL10,0.011111,-0.004762,0.290323,-0.078730
1,aupr_corrected,dataset_IL10,IL10,0.011111,-0.004762,0.290323,-0.078730
2,auroc,dataset_IL10,IL10,0.010870,-0.005003,0.274194,-0.091925


In [19]:
ligand_activity_performance_setting_summary = eval["performances_ligand_prediction"][[
    "metric",
    "aupr",
    "aupr_corrected",
    "auroc",
    "pearson"
]].groupby("metric").mean()
ligand_activity_performance_setting_summary["geom_average"] = [
    np.exp((np.log(aupr) + np.log(auroc)) / 2)
    for aupr, auroc in zip(
        ligand_activity_performance_setting_summary["aupr_corrected"],
        ligand_activity_performance_setting_summary["auroc"]
    )
]
ligand_activity_performance_setting_summary.reset_index(inplace=True)
ligand_activity_performance_setting_summary

,metric,aupr,aupr_corrected,auroc,pearson,geom_average
0,aupr,0.164179,0.149518,0.542655,0.072149,0.284845
1,aupr_corrected,0.164179,0.149518,0.542655,0.072149,0.284845
2,auroc,0.159448,0.144787,0.565183,0.055239,0.286061
3,pearson,0.176651,0.161989,0.566915,0.068570,0.303042


In [20]:
best_metric = max(
    zip(
        ligand_activity_performance_setting_summary["metric"],
        ligand_activity_performance_setting_summary["geom_average"]
    ),
    key=lambda x : x[1]
)[0]
performances_ligand_prediction_summary = eval["performances_ligand_prediction"][
    eval["performances_ligand_prediction"]["metric"] == best_metric
]
performances_ligand_prediction_summary

,metric,group,ligand,aupr,aupr_corrected,auroc,pearson
3,pearson,spectrum_Ifng_Tnf,"[TNF, IFNG]",0.500000,0.500000,NaN,NaN
3,pearson,dataset_IL1A,IL1A,0.008772,-0.007101,0.096774,-0.086320
3,pearson,dataset_TNF,TNF,0.022727,0.006854,0.709677,0.009455
3,pearson,dataset_HMGB1,HMGB1,0.009804,-0.006069,0.193548,-0.102390
3,pearson,dataset_IL36B,IL36B,0.016129,0.000256,0.516129,-0.051438
...,...,...,...,...,...,...,...
3,pearson,tnf_ltb_aorta,"[TNF, LTB]",0.500000,0.500000,NaN,NaN
3,pearson,dataset_IGF1,IGF1,1.000000,0.984127,1.000000,0.392306
3,pearson,dataset_IL21,IL21,0.012500,-0.003373,0.370968,-0.074124
3,pearson,rat_Il1b,IL1B,0.250000,0.234127,0.983871,0.438010


In [24]:
performances_ligand_prediction_averaged = [
    _average_performances(ligand, performances_ligand_prediction_summary)
    for ligand in ligands_evaluation
]

In [26]:
[
    -np.mean(performances_target_prediction_averaged),
    -(np.median(performances_ligand_prediction_averaged) + np.mean(performances_ligand_prediction_averaged)) / 2
]

[np.float64(-0.01776104944524588), np.float64(-0.09300837250206284)]

In [12]:
from optuna.trial import Trial
from optuna import create_study

def objective(trial:Trial):
    pass

study = create_study()
study.optimize(objective, n_trials=1)

c:\Users\victorm\Documents\nichenetpy\.hatch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2025-07-02 09:32:54,562] A new study created in memory with name: no-name-49723662-ee0a-4ae1-9687-50f4272d3457
[W 2025-07-02 09:32:54,563] Trial 0 failed with parameters: {} because of the following error: The value None could not be cast to float..
[W 2025-07-02 09:32:54,563] Trial 0 failed with value None.


In [13]:
session_info.show()